# 7. Gradient Descent, Learning Rates and OptimizersFrom *Transformers from Scratch*.

In [ ]:
# On Colab there is nothing on disk, so this fetches the two or# three files the chapter uses. If you cloned the repository it# finds them where they already are and downloads nothing.import osimport sysimport urllib.requestBASE = "https://raw.githubusercontent.com/joshfestus/Transformers-from-Scratch/main/code/"PLACES = [".", "code", "../code"]for name in ["ch07_optimizers.py"]:    if not any(os.path.exists(os.path.join(where, name))               for where in PLACES):        urllib.request.urlretrieve(BASE + name, name)for where in PLACES:    if os.path.isdir(where) and where not in sys.path:        sys.path.insert(0, where)

Gradient descent, learning rates and optimizers.A bowl that is much steeper in one direction than the other, which is the situation every real loss surface is in. Plain descent has to creep along the shallow direction or bounce along the steep one. This file measures what each optimizer does about that.

In [ ]:
import numpy as npSTEEP = 200.0           # curvature in the second directionSTART = np.array([-2.4, 0.9])

Half a quadratic bowl, twenty times steeper in the second axis.

In [ ]:
def loss(point):    x, y = point    return 0.5 * (x ** 2 + STEEP * y ** 2)

In [ ]:
def gradient(point):    x, y = point    return np.array([x, STEEP * y])

Plain gradient descent. Subtract the gradient, scaled.

In [ ]:
def descend(rate, steps=40, start=START):    point = np.array(start, dtype=float)    path = [point.copy()]    for _ in range(steps):        point = point - rate * gradient(point)        path.append(point.copy())    return np.array(path)

Keep a running average of past gradients and step along that.

In [ ]:
def with_momentum(rate, beta=0.9, steps=40, start=START):    point = np.array(start, dtype=float)    velocity = np.zeros_like(point)    path = [point.copy()]    for _ in range(steps):        velocity = beta * velocity + (1 - beta) * gradient(point)        point = point - rate * velocity        path.append(point.copy())    return np.array(path)

Track the average gradient and its typical size, then divide.

In [ ]:
def adam(rate, beta1=0.9, beta2=0.999, eps=1e-8, steps=40, start=START):    point = np.array(start, dtype=float)    m = np.zeros_like(point)    v = np.zeros_like(point)    path = [point.copy()]    for step in range(1, steps + 1):        g = gradient(point)        m = beta1 * m + (1 - beta1) * g        v = beta2 * v + (1 - beta2) * g ** 2        m_hat = m / (1 - beta1 ** step)        v_hat = v / (1 - beta2 ** step)        point = point - rate * m_hat / (np.sqrt(v_hat) + eps)        path.append(point.copy())    return np.array(path)

Linear warmup, then a cosine decay to nought.

In [ ]:
def schedule(step, total=1000, warmup=100, peak=1.0):    if step < warmup:        return peak * step / warmup    progress = (step - warmup) / (total - warmup)    return peak * 0.5 * (1 + np.cos(np.pi * progress))

In [ ]:
def main():    print("PLAIN DESCENT AT THREE LEARNING RATES")    print("  rate     loss after 40 steps   what happened")    for rate, note in [(0.002, "creeps along the shallow direction"),                       (0.0094, "the best this method can manage"),                       (0.0101, "bounces, and the bounce grows")]:        end = loss(descend(rate)[-1])        shown = f"{end:.4f}" if end < 1e6 else f"{end:.1e}"        print(f"  {rate:<8} {shown:<21} {note}")    print("\nTHE LARGEST RATE THAT DOES NOT DIVERGE")    print("  for this bowl it is two divided by the steepest curvature,")    print(f"  which is 2 / {STEEP:.0f} = {2 / STEEP:.2f}")    print("\nTHREE OPTIMIZERS, EACH AT ITS OWN BEST RATE, 40 STEPS")    runs = [("plain descent", descend(0.0094)),            ("momentum", with_momentum(0.060)),            ("adam", adam(0.31))]    print("  optimizer        final loss   distance from the bottom")    for name, path in runs:        end = path[-1]        away = np.linalg.norm(end)        print(f"  {name:<16} {loss(end):<12.5f} {away:.5f}")    print("\nA LEARNING RATE SCHEDULE")    print("  step    rate")    for step in [0, 25, 50, 100, 250, 500, 750, 1000]:        print(f"  {step:<7} {schedule(step):.3f}")

In [ ]:
main()

The chapter this came from works through every number above, and explains why each step is there.